# Tahap 1 + awal Tahap 2 — Latih probe + `L_group`, uji di sel yang sengaja disedikitkan

Implementasi pertama dari rumus `L_group` (`notes/research_question/03_pivot2_group_consistency.md`
§12.13), diuji di skenario sparsifikasi ala Tahap 2 (§12.7). Ini pilot pertama
yang beneran **melatih model** (bukan cuma ekstraksi & ukur kayak notebook 06/07).

**Yang dites:** 3 sel `RACExRELIG` yang aslinya datanya banyak (`White |
Protestant`, `Black | Protestant`, `White | Roman Catholic`) sengaja
"disedikitkan" datanya (subsample beneran ke N=5/10/20/50 responden asli),
lalu dibandingkan 3 cara nebak jawaban aslinya:

1. **Probe polos** -- cuma loss KL biasa, tanpa `L_group`.
2. **Probe + `L_group`** -- loss KL + suku "karet gelang" yang narik prediksi
   ke arah sel-sel tetangga yang representasinya deket (§12.13).
3. **Baseline partial-pooling/shrinkage** -- versi sederhana logika MRP
   (tarik ke arah rata-rata tetangga, kekuatan tarikan tergantung N),
   tanpa LLM sama sekali.

Kalau `L_group` beneran berguna, dia harus **lebih akurat dari probe polos**,
terutama di N kecil -- dan idealnya sebanding/lebih baik dari baseline
shrinkage (yang nggak butuh LLM sama sekali, jadi barometer "apakah LLM-nya
nambah nilai atau enggak", persis pertanyaan riset dari
`research_question/00_overview.md` §9).

## Sebelum jalan: setting Kaggle

1. **Accelerator**: GPU T4 x2 atau P100. **Internet: On**.
2. **Upload 2 Kaggle Dataset**:
   - `opinionqa_intersectional.csv` (path asli:
     `datasets/subpop/data/opinionqa/processed/opinionqa_intersectional.csv`)
     -- sama seperti notebook 07.
   - **Data mentah 5 gelombang survei**: folder
     `datasets/opinionqa_original/data/human_resp/American_Trends_Panel_W{42,49,92,45,50}/`
     (tiap folder isinya `responses.csv` + `info.csv`). Upload ke-5 folder ini
     sebagai 1 Kaggle Dataset (boleh di-zip dulu atau upload semua file
     sekaligus ke 1 dataset, asal strukturnya tetap
     `.../American_Trends_Panel_W<nomor>/responses.csv` bisa ditemukan).

Perkiraan waktu: **60-90 menit** (~15-20 menit ekstraksi representasi +
~40-50 menit latihan probe). Run sebelumnya OOM di ekstraksi (model 7B
ternyata dipadatkan ke 1 GPU doang oleh `device_map="auto"`) -- sudah
diperbaiki (model dipaksa split ke kedua T4 + batch ekstraksi diperkecil +
ada auto-retry kalau masih OOM), tapi kalau ekstraksi jadi lebih lambat dari
perkiraan karena batch kecil, itu wajar (keamanan diutamakan dulu daripada
kecepatan setelah OOM kemarin).

**Cek cell load model** (bagian 4): kalau di situ tercetak sisa memori GPU
yang masih banyak (>4 GB per GPU), `EXTRACTION_BATCH_SIZE` di cell config
bisa dinaikkan manual biar ekstraksi lebih cepat.

**Cek cell diagnostik (bagian 6b) dulu sebelum nunggu sampai selesai penuh**:
kalau di situ `KL=...` yang dicetak tiap beberapa epoch keliatan **naik**
terus (bukan turun), STOP -- ada yang salah lagi, jangan tunggu sampai akhir.
Kalau turun stabil, aman lanjut ke eksperimen penuh di bawahnya.

In [ ]:
!pip install -q -U "transformers>=4.44" accelerate scikit-learn tqdm
!pip install -q -U bitsandbytes

In [ ]:
import os
# Kurangi fragmentasi memori CUDA -- disaranin langsung oleh error OOM run
# sebelumnya. Harus di-set SEBELUM torch bikin CUDA context (makanya di baris
# paling atas, sebelum import torch).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import ast
import gc
import sys
import glob
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import pairwise_distances
from scipy.stats import wasserstein_distance
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

# PENTING kalau notebook ini pernah OOM crash di kernel yang SAMA sebelumnya
# (run ulang cell tanpa restart session): Jupyter/IPython otomatis nyimpen
# SELURUH traceback error terakhir lewat sys.last_traceback -- termasuk
# semua tensor GPU di tiap frame fungsi yang lagi jalan pas crash. Ini bikin
# memori GPU dari crash sebelumnya TIDAK PERNAH kebebasin walau modelnya
# sendiri kelihatan "sudah di-load ulang dari awal". Baris di bawah
# ini cuma jaga-jaga (bersihin apa yang BISA dibersihin di kernel yang
# sedang jalan) -- kalau memang abis crash OOM, cara paling aman & pasti
# tetap RESTART SESSION Kaggle-nya dulu (bukan cuma run ulang cell),
# baru Run All lagi dari awal.
sys.last_traceback = None
gc.collect()

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        free_b, total_b = torch.cuda.mem_get_info(i)
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} -- {free_b / 1e9:.2f} GB bebas "
              f"dari {total_b / 1e9:.2f} GB total")
        if free_b / total_b < 0.9:
            print(
                f"    !! GPU {i} sudah kepakai signifikan SEBELUM model dimuat. "
                "Kalau ini kejadian padahal baru mulai, kemungkinan besar kernel "
                "ini bekas OOM crash sebelumnya -- RESTART SESSION Kaggle dulu, "
                "baru Run All lagi dari awal."
            )
else:
    raise RuntimeError(
        "GPU tidak terdeteksi. Cek Notebook options -> Accelerator -> GPU T4 x2/P100, "
        "restart & Run All lagi."
    )

In [ ]:
# ── Model ────────────────────────────────────────────────────────
MODEL_PATH = "mistralai/Mistral-7B-v0.1"
USE_4BIT = False
# Diturunkan dari 16 -- run sebelumnya OOM karena device_map="auto" ternyata
# memadatkan SELURUH model 7B (fp16 ~14GB) ke 1 T4 saja (nyisa ~100MB), GPU
# kedua nganggur total. Sekarang model dipaksa split ke KEDUA T4 (lihat cell
# load model di bawah), jadi ada lebih banyak ruang -- tapi tetap mulai
# konservatif dulu, bisa dinaikkan lagi kalau di cell load model ternyata
# sisa memorinya banyak (dicetak di situ). Ada juga auto-retry turun batch
# kalau masih OOM (lihat fungsi ekstraksinya).
EXTRACTION_BATCH_SIZE = 4

# Balik ke CPU (kayak run pertama) buat training probe -- dites lokal, GPU
# cuma ngasih percepatan ~5% di bagian ini (bukan compute-bound, tapi banyak
# operasi kecil berurutan per pertanyaan), jadi nggak sepadan sama risiko
# berbagi memori GPU dengan proses ekstraksi model 7B yang jauh lebih butuh
# memori itu. Yang WAJIB di GPU cuma ekstraksi representasi (model 7B,
# nggak masuk akal dijalankan di CPU -- bisa berjam-jam).
PROBE_DEVICE = "cpu"

# ── Cari data ────────────────────────────────────────────────────
_candidates = glob.glob("/kaggle/input/**/opinionqa_intersectional.csv", recursive=True)
if _candidates:
    DATA_PATH = _candidates[0]
elif os.path.exists("opinionqa_intersectional.csv"):
    DATA_PATH = "opinionqa_intersectional.csv"
else:
    raise FileNotFoundError(
        "Tidak ketemu opinionqa_intersectional.csv. Upload dulu sebagai Kaggle Dataset."
    )
print("Data sel irisan dari:", DATA_PATH)

RESTRICT_WAVES = [42, 49, 92, 45, 50]
WAVE_DIRS = {}
for w in RESTRICT_WAVES:
    cands = glob.glob(f"/kaggle/input/**/American_Trends_Panel_W{w}", recursive=True)
    if not cands:
        # coba cari langsung file responses.csv-nya
        cands2 = glob.glob(f"/kaggle/input/**/American_Trends_Panel_W{w}/responses.csv", recursive=True)
        if cands2:
            cands = [os.path.dirname(cands2[0])]
    if not cands:
        raise FileNotFoundError(
            f"Tidak ketemu folder American_Trends_Panel_W{w} di /kaggle/input. "
            "Upload data mentah 5 gelombang ini sebagai Kaggle Dataset dulu."
        )
    WAVE_DIRS[w] = cands[0]
print("Wave raw data ditemukan di:")
for w, d in WAVE_DIRS.items():
    print(f"  W{w}: {d}")

# ── Skenario eksperimen ───────────────────────────────────────────
ATTR_TYPE = "RACExRELIG"
SPARSE_CELLS = ["White | Protestant", "Black | Protestant", "White | Roman Catholic"]
N_LEVELS = [5, 10, 20, 50]
N_REPEATS = 3  # diturunkan dari 5 -- epoch dinaikkan banyak, ini biar total waktu jalan wajar

LAYER_START, LAYER_END = 17, 27  # inklusif, sesuai §12.13.3
K_NEIGHBORS = 5
LAMBDA_GROUP = 1.0
N_EPOCHS = 300  # dinaikkan dari 30 -- run pertama kelihatan probe-nya belum sempat belajar
LEARNING_RATE = 0.001  # diturunkan dari 0.01 -- run pertama loss-nya DIVERGEN (naik terus),
                       # bukan sekadar lambat. Representasi mentah LLM skalanya besar (bukan
                       # angka kecil 0-1), lr=0.01 kegedean buat itu. Dicek lokal: lr=0.01
                       # divergen di skala besar, lr=0.001 stabil di semua skala yang dicoba.

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

OUT_DIR = "/kaggle/working/tahap1_lgroup_pilot"
assert not OUT_DIR.startswith("/kaggle/input"), "OUT_DIR harus di /kaggle/working!"
os.makedirs(OUT_DIR, exist_ok=True)
print("\nOUT_DIR:", OUT_DIR)
print("PROBE_DEVICE:", PROBE_DEVICE)

## 1. Siapkan data: sel `RACExRELIG`, K=2, dari 5 gelombang terpilih

In [ ]:
df = pd.read_csv(DATA_PATH)

def _parse(x):
    return ast.literal_eval(x) if isinstance(x, str) else x

df["responses"] = df["responses"].apply(_parse)
df["ordinal"] = df["ordinal"].apply(_parse)
df["options"] = df["options"].apply(_parse)

sub = df[
    (df["attribute"] == ATTR_TYPE)
    & (df["ordinal"].apply(len) == 2)
    & (df["wave"].isin(RESTRICT_WAVES))
].reset_index(drop=True)

GROUP_KEYS = sorted(sub["group"].unique().tolist())
n_g = len(GROUP_KEYS)
print(f"Total baris: {len(sub)}")
print(f"Jumlah sel: {n_g}")
print(f"Jumlah qkey (K=2, 5 wave terpilih): {sub['qkey'].nunique()}")
for c in SPARSE_CELLS:
    assert c in GROUP_KEYS, f"Sel sparsify {c} tidak ada di data terfilter!"
print("Semua sel sparsify target ada di data. OK.")

# lookup cepat: (qkey, group) -> baris
row_lookup = sub.set_index(["qkey", "group"]).to_dict("index")
qkeys_by_cell = sub.groupby("group")["qkey"].apply(set).to_dict()
QKEYS = sorted(sub["qkey"].unique().tolist())

## 2. Bikin versi "sengaja sedikit data" buat 3 sel target

Buat tiap (qkey, sel-target), tarik ulang responden ASLI dari file gelombang
mentahnya, subsample ke N=5/10/20/50 (5 kali ulang per N biar nggak
kebetulan 1 sampel aneh), hitung distribusi jawaban dari sampel kecil itu --
inilah yang dipakai buat melatih model (bukan distribusi asli).

Distribusi ASLI (dari sampel penuh, kolom `responses` di data) tetap
disimpan sebagai kunci jawaban buat ngukur akurasi nanti.

In [ ]:
wave_raw = {w: pd.read_csv(os.path.join(d, "responses.csv"), low_memory=False) for w, d in WAVE_DIRS.items()}
print("Wave mentah dimuat:", {w: len(df_) for w, df_ in wave_raw.items()})

def subsample_response(cell, qkey, n, seed):
    row = row_lookup[(qkey, cell)]
    wave = row["wave"]
    ordinal_refs = row["options"][: len(row["ordinal"])]
    race, relig = cell.split(" | ")
    weight_col = f"WEIGHT_W{wave}"
    wdf = wave_raw[wave]
    pool = wdf[(wdf["RACE"] == race) & (wdf["RELIG"] == relig) & (wdf[qkey].isin(ordinal_refs))]
    if len(pool) == 0:
        return None
    local_rng = np.random.default_rng(seed)
    idx = local_rng.choice(len(pool), size=min(n, len(pool)), replace=False)
    draw = pool.iloc[idx]
    counts = {ref: draw.loc[draw[qkey] == ref, weight_col].sum() for ref in ordinal_refs}
    total = sum(counts.values())
    if total <= 0:
        return None
    return [counts[r] / total for r in ordinal_refs]

# bangun label sparsifikasi: (cell, qkey, n, repeat) -> distribusi kecil
sparse_labels = {}
skipped = 0
for cell in SPARSE_CELLS:
    for qkey in tqdm(sorted(qkeys_by_cell[cell]), desc=f"Sparsifikasi {cell}"):
        for n in N_LEVELS:
            for rep in range(N_REPEATS):
                seed = hash((cell, qkey, n, rep)) % (2**31)
                dist = subsample_response(cell, qkey, n, seed)
                if dist is None:
                    skipped += 1
                    continue
                sparse_labels[(cell, qkey, n, rep)] = dist

print(f"\nTotal label sparsifikasi dibuat: {len(sparse_labels)} (skip: {skipped})")

## 3. Bangun prompt: kelompok (buat graf `w_ij`) & gabungan pertanyaan+kelompok (input probe)

In [ ]:
def build_group_prompt(cell):
    race, relig = cell.split(" | ")
    return f"This survey respondent's race is {race} and their religion is {relig}."

def build_combined_prompt(cell, qkey):
    row = row_lookup[(qkey, cell)]
    q, opts = row["question"], row["options"]
    letters = [chr(ord("A") + i) for i in range(len(opts))]
    lines = [build_group_prompt(cell), "", q] + [f"{l}. {o}" for l, o in zip(letters, opts)] + ["Answer:"]
    return "\n".join(lines)

print("Contoh prompt kelompok:\n", build_group_prompt(GROUP_KEYS[0]))
print("\nContoh prompt gabungan:\n", build_combined_prompt(GROUP_KEYS[0], sorted(qkeys_by_cell[GROUP_KEYS[0]])[0]))

## 4. Load model, ekstrak representasi (rata-rata layer 17-27)

In [ ]:
print(f"Loading tokenizer & model: {MODEL_PATH}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # penting: biar posisi token TERAKHIR selalu sama walau di-batch

# device_map="balanced" (BUKAN "auto") -- dipaksa split ke SEMUA GPU yang ada.
# Run sebelumnya OOM karena "auto" ternyata menaruh SELURUH model 7B (fp16
# ~14GB) di 1 T4 saja (nyisa ~100MB free), sedangkan T4 kedua nganggur total
# -- ini penyebab akar dari GPU yang "kurang kepakai" dari awal. "balanced"
# memaksa accelerate membagi rata layer ke kedua GPU walau modelnya sendiri
# muat di 1 GPU, supaya ada ruang kosong buat aktivasi batch ekstraksi.
model_kwargs = dict(torch_dtype=torch.float16, device_map="balanced", low_cpu_mem_usage=True)
if USE_4BIT:
    from transformers import BitsAndBytesConfig
    model_kwargs.pop("torch_dtype", None)
    model_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

model = AutoModelForCausalLM.from_pretrained(MODEL_PATH, **model_kwargs)
model.eval()
NUM_LAYERS = model.config.num_hidden_layers
print(f"Model loaded. Jumlah layer: {NUM_LAYERS}")
print("Sebaran model per GPU (hf_device_map):", getattr(model, "hf_device_map", "?"))
assert LAYER_END < NUM_LAYERS, "LAYER_END melebihi jumlah layer model!"

for i in range(torch.cuda.device_count()):
    free_b, total_b = torch.cuda.mem_get_info(i)
    print(f"  GPU {i}: {free_b / 1e9:.2f} GB bebas dari {total_b / 1e9:.2f} GB "
          f"(sesudah model dimuat) -- kalau ini masih banyak (>4 GB), "
          f"EXTRACTION_BATCH_SIZE di cell config bisa dinaikkan lagi.")


@torch.no_grad()
def get_avg_layer_embeddings_batch(prompts, batch_size=EXTRACTION_BATCH_SIZE):
    """Rata-rata representasi token TERAKHIR di layer LAYER_START..LAYER_END,
    diproses per-batch (bukan satu-satu) biar GPU kepakai lebih penuh.

    Pakai left-padding + position_ids eksplisit (bukan default arange), biar
    hasilnya identik sama proses satu-satu -- sudah divalidasi lokal pakai
    model kecil (hf-internal-testing/tiny-random-gpt2), beda maksimal antara
    versi batch dan versi satu-satu di angka 1e-6 (cuma pembulatan floating
    point, bukan bug).

    Kalau batch_size ini masih kegedean buat memori yang ada, tangkap
    OutOfMemoryError dan otomatis coba lagi dengan batch separuhnya -- biar
    nggak harus tebak angka pas dari awal / gagal total di tengah proses
    yang sudah jalan lama.
    """
    all_embeddings = []
    i = 0
    bs = batch_size
    while i < len(prompts):
        batch = prompts[i : i + bs]
        try:
            enc = tokenizer(batch, return_tensors="pt", padding=True).to(model.device)
            attn = enc["attention_mask"]
            position_ids = (attn.cumsum(-1) - 1).masked_fill(attn == 0, 1)

            out = model(
                input_ids=enc["input_ids"],
                attention_mask=attn,
                position_ids=position_ids,
                output_hidden_states=True,
            )
            # slice DULU ke window layer yang dipakai, baru stack -- lebih hemat
            # memori daripada nge-stack semua 33 layer trus baru diiris.
            hs_window = torch.stack(out.hidden_states[LAYER_START : LAYER_END + 1], dim=0)
            last_token = hs_window[:, :, -1, :]  # left-padded -> posisi -1 selalu token asli terakhir
            avg = last_token.mean(dim=0)  # (batch, hidden)
            all_embeddings.append(avg.float().cpu().numpy())

            del out, hs_window, last_token, avg, enc, attn, position_ids
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            i += bs
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if bs <= 1:
                raise
            bs = max(1, bs // 2)
            print(f"  [OOM] turunkan batch jadi {bs} dan ulangi dari prompt ke-{i}...")

    return np.concatenate(all_embeddings, axis=0)

In [ ]:
# representasi kelompok-saja (buat graf w_ij) -- diproses per-batch
cell_prompts_list = [build_group_prompt(c) for c in GROUP_KEYS]
cell_emb_array = get_avg_layer_embeddings_batch(cell_prompts_list)
cell_embeddings = {c: cell_emb_array[i] for i, c in enumerate(GROUP_KEYS)}
print("Bentuk representasi kelompok:", cell_emb_array.shape)

In [ ]:
# representasi gabungan pertanyaan+kelompok (input probe) -- diproses per-batch,
# 1x per (qkey, cell) yang valid
valid_pairs = [(qkey, cell) for qkey in QKEYS for cell in GROUP_KEYS if (qkey, cell) in row_lookup]
print(f"Jumlah pasangan (qkey, sel) valid: {len(valid_pairs)}")

pair_prompts_list = [build_combined_prompt(cell, qkey) for qkey, cell in valid_pairs]
pair_emb_array_raw = get_avg_layer_embeddings_batch(pair_prompts_list)
pair_embeddings = {pair: pair_emb_array_raw[i] for i, pair in enumerate(valid_pairs)}

# Normalisasi ke panjang vektor 1 (L2 norm) -- representasi mentah LLM skalanya
# besar dan nggak konsisten, kalau langsung dipakai ke probe bisa bikin latihan
# nggak stabil (kejadian di run pertama: loss malah naik/divergen, bukan turun).
# Ini pengaman TAMBAHAN di luar penurunan LEARNING_RATE di cell config -- biar
# nggak cuma bergantung nebak 1 angka learning rate yang pas.
raw_norms = np.array([np.linalg.norm(v) for v in pair_embeddings.values()])
print(f"Panjang vektor representasi SEBELUM dinormalisasi: min={raw_norms.min():.2f}, "
      f"median={np.median(raw_norms):.2f}, max={raw_norms.max():.2f}")
pair_embeddings = {k: v / np.linalg.norm(v) for k, v in pair_embeddings.items()}

# simpan JUGA representasi gabungan (bukan cuma yang kelompok-saja) -- biar kalau
# perlu debug training lagi nanti, nggak perlu ekstraksi ulang lewat GPU.
pair_qkeys = np.array([q for q, c in valid_pairs], dtype=object)
pair_cells = np.array([c for q, c in valid_pairs], dtype=object)
pair_emb_array = np.stack([pair_embeddings[(q, c)] for q, c in valid_pairs])

np.savez(
    os.path.join(OUT_DIR, "embeddings_pilot.npz"),
    cell_emb=cell_emb_array,
    group_keys=np.array(GROUP_KEYS, dtype=object),
    pair_emb=pair_emb_array,
    pair_qkeys=pair_qkeys,
    pair_cells=pair_cells,
)
print("Representasi kelompok DAN gabungan (sudah dinormalisasi) disimpan ke embeddings_pilot.npz.")

## 5. Bangun graf `w_ij` (k-NN + kernel Gaussian, median heuristic)

Karena semua sel di sini 1 tipe kombinasi (`RACExRELIG`), gerbang "beda tipe
= nol" dari §12.13.3 otomatis tidak pernah aktif -- semua pasangan
diperbolehkan ikut kernel, dibatasi cuma lewat k-tetangga-terdekat.

In [ ]:
def build_knn_graph(emb_array, k):
    norm = emb_array / np.linalg.norm(emb_array, axis=1, keepdims=True)
    cos_dist = 1 - (norm @ norm.T)
    np.fill_diagonal(cos_dist, np.inf)
    sigma = np.median(cos_dist[np.isfinite(cos_dist)])
    w = np.exp(-(cos_dist ** 2) / (2 * sigma ** 2))
    n = emb_array.shape[0]
    knn_mask = np.zeros((n, n), dtype=bool)
    for i in range(n):
        nn_idx = np.argsort(cos_dist[i])[:k]
        knn_mask[i, nn_idx] = True
    knn_mask = knn_mask | knn_mask.T
    w = np.where(knn_mask, w, 0.0)
    np.fill_diagonal(w, 0.0)
    return w, sigma

W_IJ, SIGMA = build_knn_graph(cell_emb_array, K_NEIGHBORS)
print(f"Sigma (median heuristic): {SIGMA:.4f}")
print(f"Jumlah edge tak-nol: {(W_IJ > 0).sum()} dari {n_g * (n_g - 1)} kemungkinan")

for cell in SPARSE_CELLS:
    i = GROUP_KEYS.index(cell)
    neighbors = [(GROUP_KEYS[j], W_IJ[i, j]) for j in np.argsort(-W_IJ[i]) if W_IJ[i, j] > 0][:5]
    print(f"\nTetangga terdekat '{cell}': {neighbors}")

## 6. Probe, loss `L_group` (JSD), dan training loop

In [ ]:
def jsd_matrix(pred, eps=1e-8):
    """Jensen-Shannon Divergence antar SEMUA pasangan sekaligus (matriks), bukan
    loop satu-satu -- jauh lebih cepat. pred: (n, K) -> hasil (n, n)."""
    p_i = pred.unsqueeze(1).clamp(min=eps)  # (n, 1, K)
    p_j = pred.unsqueeze(0).clamp(min=eps)  # (1, n, K)
    m = 0.5 * (p_i + p_j)
    kl_i = (p_i * (p_i / m).log()).sum(dim=-1)  # (n, n)
    kl_j = (p_j * (p_j / m).log()).sum(dim=-1)  # (n, n)
    return 0.5 * kl_i + 0.5 * kl_j


# Semua representasi gabungan dipindah jadi 1 tensor GPU sekali di sini (bukan
# dict numpy yang di-convert ulang tiap epoch) -- ini yang bikin training loop
# beneran pakai GPU, bukan CPU kayak sebelumnya (GPU nganggur pas training ~40-50
# menit itu penyebab utama "GPU kurang dipakai").
pair_index = {pair: i for i, pair in enumerate(valid_pairs)}
pair_emb_tensor = torch.tensor(pair_emb_array, dtype=torch.float32, device=PROBE_DEVICE)
W_IJ_full = torch.tensor(W_IJ, dtype=torch.float32, device=PROBE_DEVICE)
GROUP_INDEX = {c: i for i, c in enumerate(GROUP_KEYS)}


def train_probe(qkey_to_cell_labels, use_group_loss, seed=RANDOM_SEED, n_epochs=N_EPOCHS,
                 lr=LEARNING_RATE, verbose=False, log_every=None):
    """qkey_to_cell_labels: dict qkey -> {cell: distribusi target (list panjang 2)}.
    Melatih 1 probe linear (D->2) di semua qkey sekaligus, di GPU (PROBE_DEVICE).
    Return: (probe, history) -- history = list (KL rata-rata, L_group rata-rata) per epoch,
    dipakai buat ngecek apakah latihannya beneran konvergen (lihat cell diagnostik)."""
    torch.manual_seed(seed)
    D = pair_emb_tensor.shape[1]
    probe = nn.Linear(D, 2).to(PROBE_DEVICE)
    optimizer = torch.optim.Adam(probe.parameters(), lr=lr)

    # pre-hitung index/label/bobot tetangga SEKALI per qkey (bukan diulang tiap
    # epoch) -- disimpan langsung sebagai tensor GPU biar loop epoch murni operasi
    # GPU tanpa bolak-balik CPU<->GPU.
    qkey_data = {}
    for qkey, cell_labels in qkey_to_cell_labels.items():
        if len(cell_labels) < 2:
            continue
        cells_present = list(cell_labels.keys())
        idx_tensor = torch.tensor([pair_index[(qkey, c)] for c in cells_present], device=PROBE_DEVICE)
        Y = torch.tensor(np.stack([cell_labels[c] for c in cells_present]), dtype=torch.float32, device=PROBE_DEVICE)
        group_idx = [GROUP_INDEX[c] for c in cells_present]
        w_sub = W_IJ_full[group_idx][:, group_idx]
        qkey_data[qkey] = (idx_tensor, Y, w_sub)

    history = []
    for epoch in range(n_epochs):
        epoch_kl, epoch_lg = [], []
        for idx_tensor, Y, w_sub in qkey_data.values():
            X = pair_emb_tensor[idx_tensor]  # (n, D), sudah di GPU, tinggal index

            logits = probe(X)
            log_probs = torch.log_softmax(logits, dim=-1)
            pred = torch.softmax(logits, dim=-1)

            kl = (Y * (Y.clamp(min=1e-8).log() - log_probs)).sum(dim=-1).mean()
            loss = kl
            lg_value = 0.0

            if use_group_loss:
                total_w = w_sub.sum()
                if total_w > 0:
                    jm = jsd_matrix(pred)
                    l_group_term = (w_sub * jm).sum() / total_w
                    loss = loss + LAMBDA_GROUP * l_group_term
                    lg_value = l_group_term.item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_kl.append(kl.item())
            epoch_lg.append(lg_value)

        history.append((float(np.mean(epoch_kl)), float(np.mean(epoch_lg))))
        if verbose and log_every and (epoch % log_every == 0 or epoch == n_epochs - 1):
            print(f"    epoch {epoch:4d}: KL={history[-1][0]:.4f}  L_group={history[-1][1]:.4f}")

    return probe, history


@torch.no_grad()
def predict_probe(probe, qkey, cell):
    idx = pair_index[(qkey, cell)]
    X = pair_emb_tensor[idx].unsqueeze(0)
    return torch.softmax(probe(X), dim=-1).squeeze(0).cpu().numpy()

# Baseline shrinkage (partial-pooling sederhana) didefinisikan di cell
# berikutnya, setelah true_labels (dipakai sebagai "prior" dari tetangga)
# sudah disiapkan.

## 6b. Diagnostik dulu sebelum eksperimen penuh

Run pertama pilot ini probe-nya jelek banget (WD 0.40-0.48, lebih jelek dari
asal nebak). Sebelum ulang eksperimen penuh (mahal, ~30-50 menit), cek dulu 2
hal murah:

1. **Apakah loss-nya beneran turun** kalau dilatih lebih lama (300 epoch,
   bukan 30) -- dicetak tiap beberapa epoch di bawah.
2. **Apakah probe-nya bisa "connect" ke data yang dia LIHAT LANGSUNG** pas
   latihan (bukan ke sel yang disparsifikasi) -- kalau di situ aja masih
   jelek, berarti soal training belum benar (bukan soal generalisasi, apalagi
   soal `L_group`).

In [ ]:
# true_labels: (qkey, cell) -> distribusi asli (dari data, N penuh) -- dipakai
# baik di diagnostik ini maupun di eksperimen utama nanti.
true_labels = {(qkey, cell): row_lookup[(qkey, cell)]["responses"] for (qkey, cell) in valid_pairs}

print("=== Diagnostik: latih 1 probe contoh (skenario N=50, TANPA L_group), pantau loss tiap epoch ===")
n_diag, rep_diag = 50, 0
qkey_to_cell_labels_diag = {}
for qkey in QKEYS:
    entry = {}
    for cell in GROUP_KEYS:
        if (qkey, cell) not in row_lookup:
            continue
        if cell in SPARSE_CELLS:
            key = (cell, qkey, n_diag, rep_diag)
            if key in sparse_labels:
                entry[cell] = sparse_labels[key]
        else:
            entry[cell] = true_labels[(qkey, cell)]
    if len(entry) >= 2:
        qkey_to_cell_labels_diag[qkey] = entry

probe_diag, history_diag = train_probe(
    qkey_to_cell_labels_diag, use_group_loss=False, seed=999,
    log_every=max(1, N_EPOCHS // 10), verbose=True,
)

print("\n=== Cek fit: apakah probe cocok sama data yang DIA SENDIRI lihat pas latihan? ===")
normal_cells = [c for c in GROUP_KEYS if c not in SPARSE_CELLS]
sample_check = [
    (qkey, cell) for qkey in list(qkey_to_cell_labels_diag)[:20]
    for cell in normal_cells if (qkey, cell) in true_labels
][:40]
fit_wds = []
for qkey, cell in sample_check:
    pred = predict_probe(probe_diag, qkey, cell)
    ordinal = row_lookup[(qkey, cell)]["ordinal"]
    wd = wasserstein_distance(ordinal, ordinal, u_weights=np.clip(pred, 1e-8, None), v_weights=true_labels[(qkey, cell)])
    fit_wds.append(wd)
print(f"WD rata-rata di {len(fit_wds)} contoh yang DIA LATIHAN LANGSUNG: {np.mean(fit_wds):.4f}")
print("- Kalau masih tinggi (>0.2): probe belum 'connect' ke datanya -- naikin N_EPOCHS lagi atau turunin LEARNING_RATE, JANGAN lanjut ke eksperimen utama dulu.")
print("- Kalau ini udah rendah: aman lanjut ke eksperimen utama di bawah -- baru di situ soal generalisasi ke sel sparsify (dan L_group) relevan diuji.")

In [ ]:
from scipy.stats import wasserstein_distance

In [ ]:
# neighbor_prior/shrinkage_predict pakai true_labels yang sudah disiapkan di
# cell diagnostik (6b) di atas -- tetangga di sini pakai label ASLI tetangga
# (info yang memang tersedia, karena cuma 3 sel target yang kita sparsifikasi).
def neighbor_prior(qkey, cell):
    i = GROUP_KEYS.index(cell)
    neighbor_idx = np.where(W_IJ[i] > 0)[0]
    dists, weights = [], []
    for j in neighbor_idx:
        nb_cell = GROUP_KEYS[j]
        if (qkey, nb_cell) in true_labels:
            dists.append(true_labels[(qkey, nb_cell)])
            weights.append(W_IJ[i, j])
    if not dists:
        return None
    dists = np.array(dists)
    weights = np.array(weights)
    return (dists * weights[:, None]).sum(axis=0) / weights.sum()

def shrinkage_predict(sparse_dist, n_obs, qkey, cell, tau=20.0):
    prior = neighbor_prior(qkey, cell)
    if prior is None:
        return np.array(sparse_dist)
    alpha = n_obs / (n_obs + tau)
    return alpha * np.array(sparse_dist) + (1 - alpha) * prior


results = []
for n in N_LEVELS:
    for rep in range(N_REPEATS):
        # bangun label training buat skenario (n, rep) ini
        qkey_to_cell_labels = {}
        for qkey in QKEYS:
            entry = {}
            for cell in GROUP_KEYS:
                if (qkey, cell) not in row_lookup:
                    continue
                if cell in SPARSE_CELLS:
                    key = (cell, qkey, n, rep)
                    if key in sparse_labels:
                        entry[cell] = sparse_labels[key]
                    # kalau nggak ada draw valid, skip sel ini utk qkey ini
                else:
                    entry[cell] = true_labels[(qkey, cell)]
            if len(entry) >= 2:
                qkey_to_cell_labels[qkey] = entry

        probe_plain, hist_plain = train_probe(qkey_to_cell_labels, use_group_loss=False, seed=1000 + rep)
        probe_group, hist_group = train_probe(qkey_to_cell_labels, use_group_loss=True, seed=1000 + rep)
        print(
            f"N={n} repeat={rep+1}/{N_REPEATS}: "
            f"KL polos {hist_plain[0][0]:.3f}->{hist_plain[-1][0]:.3f}, "
            f"KL+L_group {hist_group[0][0]:.3f}->{hist_group[-1][0]:.3f} "
            f"(L_group akhir={hist_group[-1][1]:.3f})"
        )

        for cell in SPARSE_CELLS:
            for qkey in sorted(qkeys_by_cell[cell]):
                key = (cell, qkey, n, rep)
                if key not in sparse_labels or qkey not in qkey_to_cell_labels or cell not in qkey_to_cell_labels[qkey]:
                    continue
                true_dist = true_labels[(qkey, cell)]
                ordinal = row_lookup[(qkey, cell)]["ordinal"]

                pred_plain = predict_probe(probe_plain, qkey, cell)
                pred_group = predict_probe(probe_group, qkey, cell)
                pred_shrink = shrinkage_predict(sparse_labels[key], n, qkey, cell)

                for method, pred in [("probe_polos", pred_plain), ("probe_L_group", pred_group), ("shrinkage", pred_shrink)]:
                    wd = wasserstein_distance(ordinal, ordinal, u_weights=np.clip(pred, 1e-8, None), v_weights=true_dist)
                    results.append({"n": n, "repeat": rep, "cell": cell, "qkey": qkey, "method": method, "wd": wd})

results_df = pd.DataFrame(results)
results_df.to_csv(os.path.join(OUT_DIR, "pilot_results_raw.csv"), index=False)
print(f"\nTotal baris hasil: {len(results_df)}")

## 8. Ringkas & plot: WD vs N, per metode

Ini kurva ala AutoElicit (§8.2) -- kalau `L_group` beneran nolong, garisnya
harus di bawah `probe_polos` terutama di N kecil.

In [ ]:
summary = results_df.groupby(["n", "method"])["wd"].agg(["mean", "std", "count"]).reset_index()
print(summary.to_string(index=False))
summary.to_csv(os.path.join(OUT_DIR, "pilot_summary.csv"), index=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors = {"probe_polos": "#d62728", "probe_L_group": "#1f77b4", "shrinkage": "#2ca02c"}
for method in ["probe_polos", "probe_L_group", "shrinkage"]:
    sub_s = summary[summary["method"] == method].sort_values("n")
    ax.errorbar(sub_s["n"], sub_s["mean"], yerr=sub_s["std"], marker="o", label=method, color=colors[method], capsize=3)
ax.set_xlabel("N (jumlah responden yang disimulasikan tersedia)")
ax.set_ylabel("Wasserstein Distance ke jawaban asli (makin kecil makin baik)")
ax.set_title("Tahap 1 pilot -- akurasi vs N, 3 metode")
ax.set_xscale("log")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "pilot_wd_vs_n.png"), dpi=150)
plt.show()

print(f"\nSemua output ada di: {OUT_DIR}")

## 9. Investigasi lanjutan: kenapa `L_group` & N kelihatan nggak ngaruh?

Hasil tabel di atas menunjukkan 2 hal yang perlu digali sebelum menyimpulkan
apa-apa soal `L_group`:
1. `probe_L_group` nyaris identik dengan `probe_polos` di SEMUA level N
   (selisihnya lebih kecil dari galat acak yang wajar) -- `L_group`
   kelihatan nggak "kerja".
2. Kedua probe (polos maupun +`L_group`) TIDAK membaik walau N (jumlah data
   asli yang dipura-purakan tersedia) naik dari 5 ke 50 -- padahal
   `shrinkage` (statistik sederhana, tanpa LLM) membaik terus seiring N.

5 cek di bawah ini (murah, nggak perlu ekstraksi ulang GPU -- cuma retrain
beberapa probe kecil pakai representasi yang sudah ada) dirancang buat
mencari tahu apakah ini soal DESAIN eksperimen (probe nggak "dengar" label
sel sparse-nya sendiri) atau memang `L_group`-nya lemah:

- **A+B**: latih ulang 2 probe kecil (skenario N=5 vs N=50, ~3-5 menit) --
  bandingkan LANGSUNG prediksinya buat sel & pertanyaan yang sama, dan cek
  seberapa pas probe fit ke LABEL SPARSE-nya sendiri (bukan cuma ke label
  asli/true yang dipakai buat skor akhir).
- **C+E**: cek seberapa besar bobot graf `w_ij` yang nyambung ke 3 sel
  target, dan apakah tetangga terdekatnya (menurut representasi LLM) emang
  masuk akal secara jawaban asli (bukan cuma "deket" tapi nggak nyambung).
- **D**: paksa naikkan `LAMBDA_GROUP` sampai 100x lipat dari nilai
  aslinya (1.0) -- kalau bahkan itu nggak mengubah apa-apa, berarti
  `L_group` secara struktural nggak berpengaruh di sini (bukan cuma soal
  nilai lambda yang kurang pas).
- **F+G**: cek posisi embedding-nya SECARA GEOMETRIS -- apakah semua sel
  memang tersebar jelas di ruang representasi, atau nge-cluster jadi 1
  gumpalan (bikin kernel kurang diskriminatif)? Dan yang paling langsung:
  pakai probe **RANDOM** (belum dilatih sama sekali) buat ngukur `L_group`
  -- kalau probe acak aja udah kasih JSD kecil ke tetangga, itu bukti
  geometri representasinya sendiri yang bikin "deket", BUKAN hasil training
  `L_group` menariknya ke sana.

In [ ]:
print("=" * 70)
print("INVESTIGASI A+B: apakah prediksi probe sensitif ke N? apakah probe")
print("fit ke label sparse-nya SENDIRI?")
print("=" * 70)

def build_qkey_to_cell_labels(n, rep):
    out = {}
    for qkey in QKEYS:
        entry = {}
        for cell in GROUP_KEYS:
            if (qkey, cell) not in row_lookup:
                continue
            if cell in SPARSE_CELLS:
                key = (cell, qkey, n, rep)
                if key in sparse_labels:
                    entry[cell] = sparse_labels[key]
            else:
                entry[cell] = true_labels[(qkey, cell)]
        if len(entry) >= 2:
            out[qkey] = entry
    return out

diag_probes = {}
for n in [5, 50]:
    qtl = build_qkey_to_cell_labels(n, 0)
    probe_n, _ = train_probe(qtl, use_group_loss=False, seed=2000)
    diag_probes[n] = (probe_n, qtl)
    print(f"Selesai latih probe diagnostik N={n} (rep=0, tanpa L_group)")

# ── A: bandingkan LANGSUNG prediksi & label training di N=5 vs N=50 ──
print(f"\n--- A. Contoh konkret: sel '{SPARSE_CELLS[0]}', 5 pertanyaan pertama ---")
sample_qkeys = sorted(qkeys_by_cell[SPARSE_CELLS[0]])[:5]
for qkey in sample_qkeys:
    cell = SPARSE_CELLS[0]
    key5, key50 = (cell, qkey, 5, 0), (cell, qkey, 50, 0)
    if key5 not in sparse_labels or key50 not in sparse_labels:
        continue
    pred5 = predict_probe(diag_probes[5][0], qkey, cell)
    pred50 = predict_probe(diag_probes[50][0], qkey, cell)
    print(f"  {qkey}:")
    print(f"    label training @N=5  = {np.round(sparse_labels[key5], 2)}   -> prediksi probe = {np.round(pred5, 2)}")
    print(f"    label training @N=50 = {np.round(sparse_labels[key50], 2)}   -> prediksi probe = {np.round(pred50, 2)}")
    print(f"    (jawaban ASLI/true    = {np.round(true_labels[(qkey, cell)], 2)})")

pred_diffs, label_diffs = [], []
for cell in SPARSE_CELLS:
    for qkey in sorted(qkeys_by_cell[cell]):
        key5, key50 = (cell, qkey, 5, 0), (cell, qkey, 50, 0)
        if key5 not in sparse_labels or key50 not in sparse_labels:
            continue
        if qkey not in diag_probes[5][1] or cell not in diag_probes[5][1][qkey]:
            continue
        pred5 = predict_probe(diag_probes[5][0], qkey, cell)
        pred50 = predict_probe(diag_probes[50][0], qkey, cell)
        pred_diffs.append(np.abs(pred5 - pred50).sum())
        label_diffs.append(np.abs(np.array(sparse_labels[key5]) - np.array(sparse_labels[key50])).sum())

print(f"\nDi {len(pred_diffs)} pasangan (qkey, sel target):")
print(f"  Rata-rata selisih LABEL training (dikasih ke probe, N=5 vs N=50): {np.mean(label_diffs):.4f}")
print(f"  Rata-rata selisih PREDIKSI probe yang dihasilkan  (N=5 vs N=50): {np.mean(pred_diffs):.4f}")
print("  -> kalau selisih PREDIKSI jauh LEBIH KECIL dari selisih LABEL, artinya")
print("     probe TIDAK banyak 'mendengarkan' label sel sparse-nya sendiri --")
print("     prediksinya didominasi pola dari 23 sel lain yang labelnya stabil.")

# ── B: seberapa pas probe fit ke label sparse-nya SENDIRI (bukan true) ──
print(f"\n--- B. Seberapa pas probe fit ke LABEL SPARSE-nya sendiri (bukan true) ---")
for n in [5, 50]:
    probe_n, qtl_n = diag_probes[n]
    own_wds = []
    for cell in SPARSE_CELLS:
        for qkey in sorted(qkeys_by_cell[cell]):
            key = (cell, qkey, n, 0)
            if key not in sparse_labels or qkey not in qtl_n or cell not in qtl_n[qkey]:
                continue
            pred = predict_probe(probe_n, qkey, cell)
            ordinal = row_lookup[(qkey, cell)]["ordinal"]
            wd_own = wasserstein_distance(
                ordinal, ordinal, u_weights=np.clip(pred, 1e-8, None), v_weights=sparse_labels[key]
            )
            own_wds.append(wd_own)
    print(f"  N={n}: WD ke label sparse SENDIRI = {np.mean(own_wds):.4f} (n={len(own_wds)} pasangan)")
print("  (bandingkan sama WD training-fit sel NORMAL di diagnostik 6b, sekitar ~0.12 --")
print("   kalau angka di 3 sel target ini jauh LEBIH TINGGI dari itu, berarti probe")
print("   memang kalah bersaing merebut kapasitas model dari 23 sel lain.)")

In [ ]:
print("=" * 70)
print("INVESTIGASI C+E: seberapa besar & masuk akal 'tarikan' L_group ke 3 sel ini")
print("=" * 70)

print("\n--- C. Bobot w_ij yang nyambung ke 3 sel target (vs rata-rata semua sel) ---")
avg_total_w = W_IJ.sum(axis=1).mean()
for cell in SPARSE_CELLS:
    i = GROUP_KEYS.index(cell)
    total_w = W_IJ[i].sum()
    n_edges = int((W_IJ[i] > 0).sum())
    flag = "  <-- di bawah rata-rata!" if total_w < avg_total_w else ""
    print(f"  {cell}: total bobot tetangga = {total_w:.4f} ({n_edges} tetangga){flag}")
print(f"  Rata-rata SEMUA sel: {avg_total_w:.4f}")
print("  (kalau bobot 3 sel target ini jauh di bawah rata-rata, artinya kernel")
print("   memang nggak nemu tetangga yang 'deket' buat sel-sel ini -- L_group")
print("   nggak punya banyak bahan buat narik ke mana pun.)")

print("\n--- E. Apakah tetangga (menurut representasi LLM) itu emang mirip SECARA JAWABAN ASLI? ---")
for cell in SPARSE_CELLS:
    i = GROUP_KEYS.index(cell)
    neighbor_idx = [j for j in np.argsort(-W_IJ[i]) if W_IJ[i, j] > 0][:3]
    sample_qkey = sorted(qkeys_by_cell[cell])[0]
    true_this = true_labels.get((sample_qkey, cell))
    print(f"\n  {cell} (contoh qkey={sample_qkey}):")
    print(f"    true dist SEL INI      = {np.round(true_this, 3) if true_this is not None else 'N/A'}")
    for j in neighbor_idx:
        nb = GROUP_KEYS[j]
        true_nb = true_labels.get((sample_qkey, nb))
        diff = (
            np.abs(np.array(true_this) - np.array(true_nb)).sum()
            if (true_this is not None and true_nb is not None) else None
        )
        diff_str = f", selisih={diff:.3f}" if diff is not None else ""
        print(f"    tetangga '{nb}' (w_ij={W_IJ[i, j]:.3f}): true dist = "
              f"{np.round(true_nb, 3) if true_nb is not None else 'N/A'}{diff_str}")
print("\n  (kalau selisih true dist ke tetangga JUSTRU besar, berarti representasi LLM")
print("   nganggep sel-sel ini 'deket' padahal jawaban aslinya beda jauh -- L_group")
print("   bisa jadi malah narik ke arah yang SALAH, bukan cuma 'nggak ngaruh'.)")

In [ ]:
print("=" * 70)
print("INVESTIGASI D: kalau LAMBDA_GROUP dipaksa naik jauh, apa L_group jadi ngaruh?")
print("=" * 70)

qtl_test = diag_probes[5][1]  # skenario N=5 -- yang paling butuh bantuan L_group
_LAMBDA_BACKUP = LAMBDA_GROUP

for lam_test in [1.0, 10.0, 100.0]:
    LAMBDA_GROUP = lam_test
    probe_g, hist_g = train_probe(qtl_test, use_group_loss=True, seed=3000)
    wds = []
    for cell in SPARSE_CELLS:
        for qkey in sorted(qkeys_by_cell[cell]):
            key = (cell, qkey, 5, 0)
            if key not in sparse_labels or qkey not in qtl_test or cell not in qtl_test[qkey]:
                continue
            pred = predict_probe(probe_g, qkey, cell)
            ordinal = row_lookup[(qkey, cell)]["ordinal"]
            wd = wasserstein_distance(
                ordinal, ordinal, u_weights=np.clip(pred, 1e-8, None),
                v_weights=true_labels[(qkey, cell)],
            )
            wds.append(wd)
    print(f"  LAMBDA_GROUP={lam_test:>6}: L_group akhir={hist_g[-1][1]:.4f}, "
          f"KL akhir={hist_g[-1][0]:.4f}, WD rata-rata ke TRUE = {np.mean(wds):.4f}")

LAMBDA_GROUP = _LAMBDA_BACKUP
print(f"\n(LAMBDA_GROUP dikembalikan ke {LAMBDA_GROUP} setelah tes ini)")
print("\n-> kalau WD-nya nggak banyak berubah walau lambda dinaikin 100x, L_group")
print("   secara STRUKTURAL nggak berpengaruh di sini (bukan cuma soal nilai lambda")
print("   yang kurang pas) -- ini bedanya penting: kalau strukturnya yang salah,")
print("   menaikkan lambda di eksperimen selanjutnya nggak akan menolong.")
print("   Kalau WD-nya JUSTRU makin jelek seiring lambda naik, itu tanda L_group")
print("   menarik ke arah yang salah (lihat investigasi E di atas).")

In [ ]:
print("=" * 70)
print("INVESTIGASI F+G: gimana posisi embedding-nya SECARA GEOMETRIS? apakah")
print("sudah 'deket' dari sononya (representasi), bukan hasil training?")
print("=" * 70)

# ── F: statistik jarak (cosine distance) SEMUA pasangan sel ──
norm = cell_emb_array / np.linalg.norm(cell_emb_array, axis=1, keepdims=True)
cos_dist_all = 1 - (norm @ norm.T)
iu = np.triu_indices_from(cos_dist_all, k=1)
all_dists = cos_dist_all[iu]

print(f"\n--- F. Statistik jarak antar SEMUA {n_g} sel ({len(all_dists)} pasangan) ---")
print(f"  min={all_dists.min():.5f}  median={np.median(all_dists):.5f}  "
      f"mean={all_dists.mean():.5f}  max={all_dists.max():.5f}  std={all_dists.std():.5f}")
print(f"  SIGMA (median heuristic, dipakai di kernel) = {SIGMA:.5f}")
print("  (SIGMA = median jarak SEMUA pasangan -- jadi ~separuh pasangan otomatis")
print("   dianggap 'lumayan deket' oleh kernel, bukan cuma tetangga sejatinya.)")

spread_ratio = (all_dists.max() - all_dists.min()) / (all_dists.mean() + 1e-12)
print(f"\n  Rasio sebaran (max-min)/mean = {spread_ratio:.4f}")
print("  -> kalau KECIL (<< 1): semua sel kelihatan 'mirip' di mata LLM (representasinya")
print("     nge-cluster jadi 1 gumpalan besar, bukan tersebar jelas per ras x agama) --")
print("     'tetangga terdekat' cuma beda TIPIS dari 'yang paling jauh', kernel jadi")
print("     kurang diskriminatif walau angkanya kelihatan masuk akal di atas kertas.")

print(f"\n--- Untuk 3 sel target, jarak ke SEMUA 25 sel lain (bukan cuma top-5) ---")
for cell in SPARSE_CELLS:
    i = GROUP_KEYS.index(cell)
    dists_from_i = np.delete(cos_dist_all[i], i)
    print(f"  {cell}: min={dists_from_i.min():.5f}, median={np.median(dists_from_i):.5f}, "
          f"max={dists_from_i.max():.5f}  (SIGMA={SIGMA:.5f})")

# ── G: apakah JSD antar tetangga SUDAH kecil pakai probe RANDOM (belum dilatih)? ──
print("\n--- G. Seberapa kecil L_group pakai probe RANDOM (0 training) -- ini kunci ---")
torch.manual_seed(9999)
probe_untrained = nn.Linear(pair_emb_tensor.shape[1], 2).to(PROBE_DEVICE)
qtl_check = diag_probes[5][1]
untrained_lg = []
with torch.no_grad():
    for qkey, cell_labels in qtl_check.items():
        cells_present = list(cell_labels.keys())
        if len(cells_present) < 2:
            continue
        idx_t = torch.tensor([pair_index[(qkey, c)] for c in cells_present], device=PROBE_DEVICE)
        X = pair_emb_tensor[idx_t]
        pred = torch.softmax(probe_untrained(X), dim=-1)
        group_idx = [GROUP_INDEX[c] for c in cells_present]
        w_sub = W_IJ_full[group_idx][:, group_idx]
        total_w = w_sub.sum()
        if total_w > 0:
            jm = jsd_matrix(pred)
            untrained_lg.append(((w_sub * jm).sum() / total_w).item())

print(f"  L_group rata-rata, probe RANDOM (belum lihat data sama sekali): {np.mean(untrained_lg):.4f}")
print(f"  L_group rata-rata, probe SETELAH 300 epoch training tadi     : ~0.001-0.007 (lihat log training)")
print("\n  -> kalau nilai RANDOM ini SUDAH kecil (sepadan sama nilai setelah training),")
print("     berarti geometri representasinya sendiri yang bikin probe APAPUN (bahkan")
print("     acak) menghasilkan prediksi mirip buat tetangga -- BUKAN training L_group")
print("     yang berhasil menariknya ke sana. Ini penjelasan paling langsung kenapa")
print("     L_group 'akhir' kecil: dia emang nggak perlu kerja keras, sudah dekat dari awal.")
print("     Kalau nilai RANDOM ini justru BESAR (jauh lebih besar dari hasil training),")
print("     berarti training-nya yang berhasil mendekatkan -- tapi hasil ini kontradiksi")
print("     sama temuan WD yang nggak membaik, dan perlu digali lagi kenapa.")

## Cara baca hasilnya

- **Kalau `probe_L_group` (biru) di bawah `probe_polos` (merah) terutama di
  N kecil (5, 10)** -- itu bukti `L_group` beneran nolong pas datanya
  sedikit, sesuai hipotesis besar riset ini.
- **Bandingkan juga sama `shrinkage` (hijau)** -- itu barometer "apakah LLM
  nambah nilai atau enggak dibanding statistik murni". Kalau `probe_L_group`
  kalah dari `shrinkage`, itu temuan jujur juga (LLM belum nambah value di
  atas statistik sederhana) -- bukan berarti gagal, itu jawaban valid buat
  RQ2 (`research_question/00_overview.md` §9).
- Kalau N makin besar, ketiga metode harusnya makin mirip/makin akurat
  (karena data asli makin dominan, tarikan `L_group`/shrinkage makin lemah)
  -- itu sanity check bahwa "tarikan cuma pas dibutuhkan" (§12.2) beneran
  kejadian.

**Kalau hasilnya kayak di atas** (`probe_L_group` ≈ `probe_polos`, keduanya
KALAH dari `shrinkage`, dan nggak membaik walau N naik) -- **bagian 9 di
atas** (Investigasi A-E) itu yang menjawab KENAPA, bukan cuma APA. Hasil
investigasi itu yang jadi dasar buat mutuskan: apakah perlu perbaiki desain
`L_group`/eksperimen ini dulu, atau sudah cukup bukti buat bergeser ke
pendekatan lain -- keputusan itu didokumentasikan lagi di
`research_question/03_pivot2_group_consistency.md` setelah hasilnya dibahas.

**Langkah selanjutnya:**
1. Download `/kaggle/working/tahap1_lgroup_pilot/`.
2. Kalau hasilnya menjanjikan: perluas ke 6 kombinasi atribut (bukan cuma
   RACExRELIG), lebih banyak sel sparsify, dan mulai coba nilai `lambda`/`k`
   yang beda-beda.
3. Kalau hasilnya bagus & stabil: pindahkan logika training ini
   (`train_probe`, `jsd`, `build_knn_graph`, `shrinkage_predict`) ke
   `scripts/` sebagai modul `.py` yang reusable, biar notebook berikutnya
   tinggal `import` -- bukan salin-tempel kayak yang sempat kejadian antara
   notebook 06 dan 07.